# 22 · E3: enhancers trained to please a predictor → Table 3

Each GRPO-trained enhancer (reward = R0, R3, R2 or a baseline predictor; 3 seeds) and the pre-trained enhancer
process the VoiceBank-DEMAND test set, which is then scored by every reward predictor and by independent measures
(PESQ, ESTOI, SI-SDR, DNSMOS, NISQA, UTMOS, WER, speaker similarity). Table 3 reports changes relative to the
pre-trained enhancer: a reward that rises while the independent measures fall is being hacked.

No training: enhancers and predictors are loaded from the release. Recompute cost: 4 GPU jobs of about 15 minutes.

In [1]:
import os, sys
from pathlib import Path

# Works when started from the repo root or from notebooks/; needs the project environment
# (`source scripts/env.sh`, kernel "hackpatch" — see 00_setup.ipynb).
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import Markdown, display
from hackpatch import repro

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200)

In [2]:
# Parameters (can also be set through environment variables when run non-interactively)
# SOURCE = "recompute": re-run the evaluation from the released checkpoints (needs GPUs), then aggregate.
# SOURCE = "released" : aggregate the released outputs of the original runs (CPU only, seconds);
#                       this rewrites the committed analysis/out/ and paper/ files, which should come out identical.
SOURCE = os.environ.get("NB_SOURCE", "recompute")
# RUN = "slurm": one GPU job per run (sbatch, account PAS2966), this kernel waits; "local": run in this kernel.
RUN = os.environ.get("NB_RUN", "slurm")

In [3]:
reg = repro.registry()
KEYS = [k for k, v in reg.items() if v["kind"] in ['e3_eval', 'e3_eval_r2']]
print(len(KEYS), "runs:", ", ".join(KEYS))

4 runs: e3_eval_s0, e3_eval_s1, e3_eval_s2, e3_eval_R2_s0


In [4]:
if SOURCE == "recompute":
    # Idempotent: runs whose recomputed output already exists are skipped.
    repro.run_keys(KEYS, "recompute", RUN)
dirs = repro.run_analysis(SOURCE)
OUT, PAPER = dirs["out"], dirs["paper"]

[analysis] runs=/fs/scratch/PAS2301/alialavi/cache/audio_assessment_hackability_release/runs
[analysis] out=/users/PAS2301/alialavi/projects/audio_assessment_hackability/analysis/out
[analysis] paper=/users/PAS2301/alialavi/projects/audio_assessment_hackability/paper


In [5]:
# Which outputs the tables below were built from (recomputed from checkpoints vs original runs).
used = repro.load_registry(repro.results_registry(SOURCE))
pd.Series({k: used[k].get("source", "released") for k in KEYS}).value_counts()

released    4
Name: count, dtype: int64

## Table 3

In [6]:
print((PAPER / "tables" / "table3.tex").read_text())

% generated by analysis/paper_assets.py
\begin{tabular}{l c c c}
\toprule
Measure & $R_0$ & $R_3$ & random-latent pairs \\
\midrule
$\Delta$ own reward & 0.263$\pm$0.010 & 0.269$\pm$0.028 & 0.334$\pm$0.051 \\
$\Delta$PESQ & -0.226$\pm$0.170 & -0.034$\pm$0.126 & 0.019$\pm$0.050 \\
$\Delta$ESTOI & -0.057$\pm$0.010 & -0.038$\pm$0.013 & -0.043$\pm$0.006 \\
$\Delta$SI-SDR (dB) & -6.1$\pm$1.4 & -5.8$\pm$1.4 & -5.7$\pm$0.6 \\
$\Delta$DNSMOS & -0.075$\pm$0.031 & -0.076$\pm$0.056 & -0.075$\pm$0.043 \\
$\Delta$NISQA & 0.335$\pm$0.186 & 0.300$\pm$0.072 & 0.468$\pm$0.059 \\
$\Delta$UTMOS & 0.018$\pm$0.006 & 0.057$\pm$0.030 & 0.076$\pm$0.011 \\
\bottomrule
\end{tabular}



In [7]:
display(Markdown((OUT / "table3_e3.md").read_text()))

| reward          | reward_rise         | dnsmos_ovrl_delta    | nisqa_mos_delta     | utmos_delta         | spk_sim_delta        | wer_delta           | pesq_delta           | estoi_delta          | sisdr_delta          | n_external_worse    |
|:----------------|:--------------------|:---------------------|:--------------------|:--------------------|:---------------------|:--------------------|:---------------------|:---------------------|:---------------------|:--------------------|
| R0              | 0.263 ± 0.010 (n=3) | -0.075 ± 0.031 (n=3) | 0.335 ± 0.186 (n=3) | 0.018 ± 0.006 (n=3) | -0.030 ± 0.003 (n=3) | 0.005 ± 0.004 (n=3) | -0.226 ± 0.170 (n=3) | -0.057 ± 0.010 (n=3) | -6.128 ± 1.390 (n=3) | 2.333 ± 0.577 (n=3) |
| R2              | 0.378 (n=1)         | -0.079 (n=1)         | 0.443 (n=1)         | 0.079 (n=1)         | -0.032 (n=1)         | 0.009 (n=1)         | -0.034 (n=1)         | -0.033 (n=1)         | -5.921 (n=1)         | 3.000 (n=1)         |
| R3              | 0.269 ± 0.028 (n=3) | -0.076 ± 0.056 (n=3) | 0.300 ± 0.072 (n=3) | 0.057 ± 0.030 (n=3) | -0.025 ± 0.006 (n=3) | 0.006 ± 0.001 (n=3) | -0.034 ± 0.126 (n=3) | -0.038 ± 0.013 (n=3) | -5.824 ± 1.356 (n=3) | 2.333 ± 0.577 (n=3) |
| baseline_noise  | 0.325 ± 0.039 (n=3) | -0.097 ± 0.030 (n=3) | 0.392 ± 0.157 (n=3) | 0.036 ± 0.025 (n=3) | -0.029 ± 0.012 (n=3) | 0.004 ± 0.003 (n=3) | -0.135 ± 0.052 (n=3) | -0.050 ± 0.005 (n=3) | -7.187 ± 0.153 (n=3) | 2.000 ± 0.000 (n=3) |
| baseline_pgd    | 0.426 ± 0.022 (n=3) | -0.095 ± 0.037 (n=3) | 0.434 ± 0.079 (n=3) | 0.043 ± 0.022 (n=3) | -0.028 ± 0.005 (n=3) | 0.007 ± 0.001 (n=3) | -0.078 ± 0.066 (n=3) | -0.051 ± 0.006 (n=3) | -7.096 ± 0.968 (n=3) | 2.667 ± 0.577 (n=3) |
| baseline_random | 0.334 ± 0.051 (n=3) | -0.075 ± 0.043 (n=3) | 0.468 ± 0.059 (n=3) | 0.076 ± 0.011 (n=3) | -0.031 ± 0.005 (n=3) | 0.007 ± 0.002 (n=3) | 0.019 ± 0.050 (n=3)  | -0.043 ± 0.006 (n=3) | -5.715 ± 0.565 (n=3) | 2.333 ± 0.577 (n=3) |

In [8]:
print("\n".join(l for l in (OUT / "stats.md").read_text().split("\n\n") if l.startswith("H3")))

H3: degraded external measures (mean over seeds) R0-reward 2.33 vs R3-reward 2.33; reward rise +0.263 / +0.269; supported = False


In [9]:
# Every number the paper quotes from this experiment: committed value vs this notebook's value.
cmp = repro.compare_numbers(OUT / "paper_numbers.json", prefixes=('e3_',))
print(f"{len(cmp)} numbers; max |diff| = {cmp.abs_diff.max():.4f}; "
      f"identical: {(cmp.abs_diff == 0).sum()}; |diff| <= 0.01: {(cmp.abs_diff <= 0.01).sum()}")
cmp.head(25)

63 numbers; max |diff| = 0.0000; identical: 63; |diff| <= 0.01: 63


,paper,reproduced,abs_diff
e3_R0_estoi_delta,-0.057,-0.057,0.0
e3_R0_estoi_delta_2dp,-0.060,-0.060,0.0
e3_R0_n_external_worse,2.330,2.330,0.0
e3_R0_n_external_worse_2dp,2.330,2.330,0.0
e3_R0_pesq_delta,-0.226,-0.226,0.0
e3_R0_pesq_delta_2dp,-0.230,-0.230,0.0
e3_R0_reward_rise,0.263,0.263,0.0
e3_R0_reward_rise_2dp,0.260,0.260,0.0
e3_R0_sisdr_delta,-6.100,-6.100,0.0
e3_R0_sisdr_delta_2dp,-6.130,-6.130,0.0
